# Template Model

## Cara memakai notebook ini

Notebook ini sudah menyiapkan **data** dan **penilaian** untukmu. Tugasmu hanya satu: **mengisi modelnya**.

1. **Salin dulu** file ini, jangan edit aslinya. Beri nama sesuai modelmu, misalnya `01_linear_regression.ipynb`, `02_random_forest.ipynb`, atau `03_xgboost.ipynb`.
2. Cari sel bertanda **`# === ISI BAGIAN INI ===`**. Hanya sel itu yang boleh kamu ubah.
3. Di sel itu, ganti `NAMA_MODEL` dengan nama modelmu, lalu hapus tanda `#` di depan **satu** baris contoh model yang sesuai.
4. Jalankan semua sel dari atas: menu **Run → Run All Cells** (atau *Restart & Run All*).
5. Lihat tabel di bagian paling bawah. Modelmu harus mengalahkan baseline terkuat di setiap skenario: `baseline_lag1` untuk `dengan_lag`, `baseline_rata_sejenis` untuk `tanpa_riwayat_sendiri`, dan `baseline_rata_kelompok` untuk `tanpa_lag`.

## Aturan

- **Jangan ubah sel bertanda `# === JANGAN DIUBAH ===`.** Sel itu menjamin ketiga model memakai data dan cara penilaian yang sama. Kalau diubah, hasilmu tidak bisa dibandingkan dengan milik anggota lain.
- **Jangan pernah memakai `X_test` / `y_test` untuk memilih pengaturan model.** Data 2025 hanya untuk penilaian akhir. Untuk mencoba-coba pengaturan, pakai `lipatan_waktu()` (lihat bagian *Opsional* di bawah).
- Setiap model otomatis dinilai pada **6 konfigurasi**: 3 skenario (`dengan_lag`, `tanpa_lag`, `tanpa_riwayat_sendiri`) × 2 versi target (`asli`, `log`). Kamu tidak perlu menulis perulangannya.
- Pada `tanpa_lag` dan `tanpa_riwayat_sendiri`, model dilatih **5 kali** per konfigurasi. Setiap kali, sebagian prodi disembunyikan dari data latih lalu ditebak, seperti prodi yang benar-benar baru. Jadi wajar kalau dua skenario ini lebih lama.
- Hasil tersimpan otomatis di `hasil/evaluasi.csv`. Selama `NAMA_MODEL` belum diganti, hasil **tidak** disimpan.

## Kalau muncul error

Baca baris **terakhir** pesan error. Sebagian besar sudah ditulis dalam Bahasa Indonesia oleh `fondasi.py` dan menjelaskan apa yang salah. Kalau masih bingung, kirim pesan error lengkapnya ke tim.

Aturan lengkap ada di `docs/STANDAR_PENGERJAAN.md`.

In [1]:
# === JANGAN DIUBAH ===
import sys
from pathlib import Path

# Cari folder akar proyek (yang berisi folder src/), lalu daftarkan src/
AKAR = Path.cwd()
while not (AKAR / "src" / "fondasi.py").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "src"))

import pandas as pd
from fondasi import (muat_data, evaluasi, info_baris, lipatan_waktu,
                     ke_log, dari_log, log_fitur, latih_dan_prediksi, SKENARIO,
                     FILE_EVALUASI, RANDOM_STATE, VERSI_FONDASI)

DATA = {}
for skenario in SKENARIO:
    for jalur in (None, "SNBP", "SNBT"):
        DATA[(skenario, jalur)] = muat_data(skenario, jalur, verbose=False)

pd.DataFrame(
    [(sk, jl or "gabungan", len(d[0]), len(d[1]), d[0].shape[1]) for (sk, jl), d in DATA.items()],
    columns=["skenario", "jalur", "baris_latih", "baris_uji", "jumlah_fitur"])

,skenario,jalur,baris_latih,baris_uji,jumlah_fitur
0,dengan_lag,gabungan,21507,8508,76
1,dengan_lag,SNBP,10758,4254,73
2,dengan_lag,SNBT,10749,4254,74
3,tanpa_lag,gabungan,32959,9799,71
4,tanpa_lag,SNBP,16480,4890,68
5,tanpa_lag,SNBT,16479,4909,69
6,tanpa_riwayat_sendiri,gabungan,22453,8834,73
7,tanpa_riwayat_sendiri,SNBP,11216,4409,70
8,tanpa_riwayat_sendiri,SNBT,11237,4425,71


In [2]:
# === ISI BAGIAN INI ===
# Duha: konfigurasi awal dari panduan; ditetapkan sebelum evaluasi 2025.
# Tidak ada tuning atau early stopping pada data uji.
NAMA_MODEL = "xgboost"
JALUR_DIPAKAI = [None]
from xgboost import XGBRegressor

def buat_model(target):
    return XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


### Opsional · Menyetel pengaturan model (hyperparameter)

Kalau ingin mencari pengaturan terbaik, **selalu** pakai `cv=lipatan_waktu(X_train)`, jangan `cv=5`. `cv=5` mengacak data sehingga model belajar dari masa depan dan skornya terlalu bagus. Untuk `tanpa_lag` dan `tanpa_riwayat_sendiri`, `lipatan_waktu()` otomatis juga menyembunyikan prodi yang sedang divalidasi dari data latihnya. Contoh (salin ke sel baru di atas sel *JANGAN DIUBAH* berikutnya):

```python
from sklearn.model_selection import GridSearchCV
X_train, X_test, y_train, y_test = DATA[("dengan_lag", None)]
cari = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid={"n_estimators": [200, 400], "max_depth": [None, 20]},
    cv=lipatan_waktu(X_train), scoring="neg_mean_absolute_error")
cari.fit(X_train, y_train)
print(cari.best_params_)   # lalu pakai angka ini di buat_model()
```

In [3]:
# === JANGAN DIUBAH ===
SIMPAN = NAMA_MODEL != "isi_nama_model"
if not SIMPAN:
    print("NAMA_MODEL belum diganti -> hasil hanya ditampilkan, TIDAK disimpan.\n")

for skenario in SKENARIO:
    for jalur in JALUR_DIPAKAI:
        X_train, X_test, y_train, y_test = DATA[(skenario, jalur)]
        for target in ("asli", "log"):
            # Skenario prodi baru: 5 lipatan, prodi uji tidak ikut dilatih (K-24).
            y_latih = y_train if target == "asli" else ke_log(y_train)
            prediksi = latih_dan_prediksi(lambda: buat_model(target), X_train, y_latih,
                                          X_test, skenario)
            if target == "log":
                prediksi = dari_log(prediksi)
            evaluasi(y_test, prediksi, NAMA_MODEL, skenario, jalur, target=target,
                     n_latih=len(X_train), n_fitur=X_train.shape[1], simpan=SIMPAN)
            print()

[evaluasi] xgboost | dengan_lag | gabungan | target=asli
  MAE=70.23  MAPE=33.61% (0 baris peminat=0 dikecualikan)  R2=0.9312  RMSLE=- (ada prediksi negatif)
  PERHATIAN: 124 prediksi bernilai negatif



[evaluasi] xgboost | dengan_lag | gabungan | target=log
  MAE=68.67  MAPE=24.59% (0 baris peminat=0 dikecualikan)  R2=0.9371  RMSLE=0.3615



[evaluasi] xgboost | tanpa_lag | gabungan | target=asli
  MAE=185.86  MAPE=146.25% (0 baris peminat=0 dikecualikan)  R2=0.6279  RMSLE=- (ada prediksi negatif)
  PERHATIAN: 491 prediksi bernilai negatif



[evaluasi] xgboost | tanpa_lag | gabungan | target=log
  MAE=171.16  MAPE=71.56% (0 baris peminat=0 dikecualikan)  R2=0.6165  RMSLE=0.7130



[evaluasi] xgboost | tanpa_riwayat_sendiri | gabungan | target=asli
  MAE=137.54  MAPE=104.25% (0 baris peminat=0 dikecualikan)  R2=0.7153  RMSLE=- (ada prediksi negatif)
  PERHATIAN: 414 prediksi bernilai negatif



[evaluasi] xgboost | tanpa_riwayat_sendiri | gabungan | target=log
  MAE=129.35  MAPE=49.78% (0 baris peminat=0 dikecualikan)  R2=0.7190  RMSLE=0.5763



In [4]:
# === JANGAN DIUBAH ===
# Hasil modelmu dibandingkan dengan baseline
if FILE_EVALUASI.exists():
    hasil = pd.read_csv(FILE_EVALUASI)
    tampil = hasil[hasil["nama_model"].isin([NAMA_MODEL, "baseline_lag1", "baseline_rata_kelompok",
                                             "baseline_rata_sejenis"])
                   & (hasil["versi_fondasi"] == VERSI_FONDASI)
                   & hasil["jalur"].isin([j or "gabungan" for j in JALUR_DIPAKAI])]
    tampil = tampil[["nama_model", "skenario", "jalur", "target", "MAE", "MAPE", "R2", "RMSLE"]]
    display(tampil.sort_values(["skenario", "jalur", "MAE"]).reset_index(drop=True))
else:
    print("hasil/evaluasi.csv belum ada. Jalankan notebooks/00_baseline.ipynb terlebih dahulu.")

,nama_model,skenario,jalur,target,MAE,MAPE,R2,RMSLE
0,baseline_lag1,dengan_lag,gabungan,asli,66.11,23.65,0.9480,0.3633
1,xgboost,dengan_lag,gabungan,log,68.67,24.59,0.9371,0.3615
2,xgboost,dengan_lag,gabungan,asli,70.23,33.61,0.9312,NaN
3,baseline_rata_kelompok,dengan_lag,gabungan,asli,304.02,570.51,0.1468,1.4271
4,xgboost,tanpa_lag,gabungan,log,171.16,71.56,0.6165,0.7130
5,xgboost,tanpa_lag,gabungan,asli,185.86,146.25,0.6279,NaN
6,baseline_rata_kelompok,tanpa_lag,gabungan,asli,299.57,610.13,0.1296,1.4581
7,xgboost,tanpa_riwayat_sendiri,gabungan,log,129.35,49.78,0.7190,0.5763
8,xgboost,tanpa_riwayat_sendiri,gabungan,asli,137.54,104.25,0.7153,NaN
9,baseline_rata_sejenis,tanpa_riwayat_sendiri,gabungan,asli,254.52,382.57,0.3087,1.2352
